# Step7 — Índice de Magnitud ENSO (Ecuador, 2015–2025)

**Proyecto:** HCI Ecuador (Hazard Chronicity Index)
**Fase:** 7 — Índice de Magnitud ENSO, por provincia y por cantón
**Autor:** Jerson Achicanoy (FAO OER) · implementación asistida
**Fecha:** 2026-08-05

**Objetivo de este notebook:** construir el Índice de Magnitud ENSO (Fase 0
§9: "traducir Débil/Moderado/Fuerte/Muy Fuerte en valores normalizados"),
usando ICEN — ya confirmado como índice ancla del proyecto en la Fase 5
(§4.2: r=0.997 contra la SST cruda de la región Niño 1+2).

**Decisión metodológica clave, no trivial:** ICEN es un índice
nacional/mensual (un solo valor por mes para todo el país), pero
Severidad, Recurrencia y Persistencia (Fases 4-6) varían por cantón. Para
que los cuatro índices sean combinables en el HCI (Fase 8), este notebook
convierte ICEN en un valor **específico por cantón**.

**Actualización 2026-08-05 (misma tarde, tras revisar el primer resultado):**
la agregación por **máximo** (elegida inicialmente) resultó con muy bajo
poder de discriminación — las 24 provincias empataban exactamente en 0.75.
Se probaron 3 alternativas con datos reales antes de decidir (ver
`docs/methodology/07_enso_magnitude_index.md` §6.1): promedio en vez de
máximo (157/221 valores distintos, correlación con Severidad de solo
0.135 — señal propia), máximo sobre ICEN continuo (mejora parcial, sigue
saturado), y promedio ponderado por severidad (buena dispersión, pero se
aleja de "traducir categorías" que pide la Fase 0). Jerson decidió calcular
**ambas** agregaciones (máximo y promedio) como columnas separadas, y
diferir la elección final del HCI compuesto al análisis de sensibilidad de
la Fase 8 — no se descarta ninguna todavía.

**English:** ENSO Magnitude Index

Translates the strength of El Nino episodes (weak to extraordinary) into a 0-1 value per canton and province, using both the maximum and the mean episode strength each unit faced.

Outputs: `enso_magnitude_index_*` tables in `data/processed/`.

## 0. Configuración del entorno

**English:** 0. Setup

In [ ]:
# Fase 7 tampoco requiere GEE ni CDS.

# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent  # notebook vive en <repo>/notebooks/
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper to create output folders
from src.utilidades.io_utils import ensure_dir

# Tabular data handling
import pandas as pd  # tablas
# Numerical operations
import numpy as np  # operaciones numéricas
# Efficient reader for GeoPackage layers
import pyogrio  # lectura eficiente de hazard_event_geo.gpkg

# Show up to 40 columns when printing DataFrames
pd.set_option("display.max_columns", 40)

In [ ]:
# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)

# Create a logger that writes to the logs folder
logger = get_logger("step7_enso_magnitude_index", config, log_dir=PROJECT_ROOT / "logs")
logger.info("Fase 7 iniciada — configuración cargada desde %s", CONFIG_PATH)

# ENSO magnitude settings from the config
MAG_CFG = config["enso_magnitude_index"]
ANCHOR_COL = MAG_CFG["anchor_enso_index"]  # "ICEN"
# Mapping from ENSO category to an ordinal value between 0 and 1
CATEGORY_VALUES = MAG_CFG["category_values"]  # dict categoría ICEN -> valor 0-1
anchor_category_col = f"{ANCHOR_COL.lower()}_category"  # 'icen_category'
# Log the anchor index and the category values
logger.info("Índice ancla: %s | valores de categoría: %s", ANCHOR_COL, CATEGORY_VALUES)

## 1. Magnitud ENSO combinada (unión ICEN + ONI, calculada en Fase 5)

Categorías oficiales ENFEN (workbook, hoja `Notas Metodológicas`) para ICEN, y
categorías estándar NOAA/CPC para ONI. Neutro y La Niña (cualquier magnitud) se
traducen a **0**: no son "El Niño", y el proyecto mide inundaciones asociadas
específicamente a El Niño — misma decisión de alcance que la definición de
"evento ENSO" de la Fase 5 (criterio ampliado, unión ICEN+ONI).

| Categoría | Valor |
|---|---|
| Neutro / La Niña (cualquier magnitud) | 0.00 |
| El Niño - Débil | 0.25 |
| El Niño - Moderado | 0.50 |
| El Niño - Fuerte | 0.75 |
| El Niño - Extraordinario (ICEN) / Muy Fuerte (ONI) | 1.00 |

A partir de esta fase, la magnitud mensual **ya no se deriva solo de ICEN**.
Fase 5 calculó `magnitud_enso_combinada = max(valor_ordinal_ICEN,
valor_ordinal_ONI)` para cada uno de los 132 meses, como parte del criterio
ENSO ampliado (unión ICEN+ONI, ver Fase 5 §5) — Fase 7 consume directamente esa
columna en vez de recalcularla desde `icen_category`. Esto es necesario porque
ONI sí registró meses "El Niño - Muy Fuerte" (el evento 2015-2016) que ICEN, en
su escala, no etiquetó como "Extraordinario" — usar solo ICEN subestimaría la
magnitud de ese episodio.

Se usa una escala **ordinal fija** (no min-max sobre el valor numérico) porque
Fase 0 pide explícitamente traducir las *categorías* (Débil/Moderado/Fuerte/Muy
Fuerte), no el valor continuo — y porque ambos índices ya tienen categorías
oficiales publicadas, no hay necesidad de inventar un esquema propio.

**English:** 1. Combined ENSO magnitude (ICEN + ONI)

In [ ]:
# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Load the extended monthly ENSO series with the combined magnitude
enso_monthly = pd.read_parquet(processed_dir / "enso_index_monthly_extended.parquet")
logger.info("enso_index_monthly_extended (Fase 5) cargado: %d filas", len(enso_monthly))

assert "magnitud_enso_combinada" in enso_monthly.columns, "Fase 5 debe exportar magnitud_enso_combinada (union ICEN+ONI)"
# Use the combined magnitude as the monthly ENSO magnitude
enso_monthly["magnitud_enso"] = enso_monthly["magnitud_enso_combinada"]
# Count months without a magnitude value
n_sin_valor = enso_monthly["magnitud_enso"].isna().sum()
assert n_sin_valor == 0, "Fase 5 no deberia dejar meses sin magnitud combinada asignada"
logger.info("Magnitud ENSO (union ICEN+ONI, calculada en Fase 5) asignada a los 132 meses; sin categorias sin mapear")

# Show the magnitude range per ENSO type
print(enso_monthly.groupby("tipo_enso")["magnitud_enso"].agg(["min", "max", "count"]).sort_values("max"))


**Nota sobre el máximo alcanzado en esta ventana:** con el criterio ampliado
(unión ICEN+ONI), 8 de los 132 meses 2015-2025 sí alcanzan magnitud **1.00**
("El Niño - Muy Fuerte" según ONI, todos dentro del episodio 2015-2016) — a
diferencia de la versión solo-ICEN de este análisis, donde el máximo observado
era "El Niño - Fuerte" (0.75) y ningún mes llegaba a 1.0. Esto es correcto y
esperado: el evento 2015-2016 fue uno de los tres más fuertes registrados
globalmente desde 1950, y ONI (índice estándar NOAA/CPC) sí lo captura como
"Muy Fuerte" aunque la escala ICEN, calibrada específicamente a la costa
peruano-ecuatoriana, lo clasifique distinto. Se documenta aquí para que el
salto de rango (0.75 → 1.00) entre la versión anterior de este notebook y la
actual no se lea como un error.

## 2. Carga de datos de eventos y nombres oficiales

Mismo patrón de las fases anteriores.

**English:** 2. Load events and official names

In [ ]:
# Load the hazard events table without geometry
he = pyogrio.read_dataframe(processed_dir / "hazard_event_geo.gpkg", read_geometry=False)
# Log the number of events loaded
logger.info("hazard_event_geo cargado: %d filas", len(he))

# Load the official canton and province names
adm2 = pyogrio.read_dataframe(
    processed_dir / "admin_boundaries.gpkg", layer="adm2_canton", read_geometry=False
)[["canton_code", "canton_name_official", "province_name_official"]].drop_duplicates()
# Remember the row count to detect accidental changes
n_antes = len(he)
# Attach official canton and province names to each event
he = he.drop(columns=["province_name_official"]).merge(adm2, on="canton_code", how="left")
# Check the join did not change the number of events
assert len(he) == n_antes
# Check every event received an official name
assert he[["canton_name_official", "province_name_official"]].isna().sum().sum() == 0
# Log that official names were attached by canton code
logger.info("Nombres oficiales incorporados por atributo (canton_code); cobertura 100%%")

# Se une la magnitud ENSO de cada mes a cada evento, por (year, month).
# Attach the ENSO magnitude of the month to each event
he = he.merge(enso_monthly[["year", "month", "magnitud_enso"]], on=["year", "month"], how="left")
# Check every event received a magnitude
assert he["magnitud_enso"].isna().sum() == 0, "todo evento 2015-2025 debe tener una magnitud ENSO asignada"

# Columns that identify a canton
CANTON_COLS = ["province_code", "province_name_official", "canton_code", "canton_name_official"]
# Columns that identify a province
PROVINCE_COLS = ["province_code", "province_name_official"]

## 3. Agregación por cantón/provincia: máximo Y promedio, como columnas separadas

Se calculan **ambas** agregaciones entre los meses afectados de cada
unidad administrativa (de la vista correspondiente: un `hazard_type` o el
total combinado `"TODOS"`):

- `enso_magnitude_max`: el `magnitud_enso` más alto — "¿al El Niño más
  fuerte que ha enfrentado este cantón?" (elección original).
- `enso_magnitude_mean`: el promedio de `magnitud_enso` — "en promedio,
  ¿qué tan fuerte es el El Niño que golpea a este cantón?" (alternativa que
  mostró mucho mejor poder de discriminación en la comparación empírica:
  157/221 valores distintos vs. prácticamente todos empatados con máximo,
  y correlación baja con `severity_index`, 0.135 — señal propia, no
  redundante).

Si ninguno de los eventos de la unidad cayó en un mes El Niño, ambos
resultados son 0 — coherente con la codificación de la sección 1 (Neutro/
La Niña = 0). **Ninguna de las dos se descarta aquí** — la elección final
para el HCI compuesto queda para el análisis de sensibilidad de la Fase 8.

**English:** 3. Aggregation per canton and province: maximum and mean as separate columns

In [ ]:
# Compute the magnitude index for one administrative level
def ejecutar_nivel_magnitud(df: pd.DataFrame, admin_cols: list[str]) -> pd.DataFrame:
    """Calcula el Índice de Magnitud ENSO por unidad administrativa (máximo
    Y promedio, como columnas separadas), desglosado por hazard_type y
    combinado ('TODOS'), con cobertura completa del universo de unidades
    (mismo principio de Fase 5 §4.6 / Fase 6 §3: una unidad sin eventos en
    una vista dada igual aparece, con ambos índices en 0).

    Args:
        df: hazard_event_geo con 'magnitud_enso' ya incorporada por evento.
        admin_cols: columnas del nivel administrativo (cantón o provincia).

    Returns:
        DataFrame con enso_magnitude_max, enso_magnitude_mean (0-1 cada
        una) y n_eventos por unidad y vista.

    English:
    Compute the ENSO magnitude index for one administrative level, per hazard and combined.

    Args:
        df: Event table with the monthly ENSO magnitude.
        admin_cols: Columns that define the administrative level.

    Returns:
        Table with the maximum and mean magnitude per unit and view, including units without events (zeros).
    """
    # Universe of all units that have any historical event
    universo = df[admin_cols].drop_duplicates()

    # Compute the magnitude index for one hazard view
    def _por_vista(sub: pd.DataFrame, label: str) -> pd.DataFrame:
        # Maximum and mean magnitude, and event count, per unit
        agg = sub.groupby(admin_cols).agg(
            enso_magnitude_max=("magnitud_enso", "max"),
            enso_magnitude_mean=("magnitud_enso", "mean"),
            n_eventos=("event_id", "count"),
        ).reset_index()
        # Add units with no events in this view
        agg = universo.merge(agg, on=admin_cols, how="left")
        # Missing maximum magnitude becomes zero
        agg["enso_magnitude_max"] = agg["enso_magnitude_max"].fillna(0.0)
        # Missing mean magnitude becomes zero
        agg["enso_magnitude_mean"] = agg["enso_magnitude_mean"].fillna(0.0)
        # Missing event count becomes zero
        agg["n_eventos"] = agg["n_eventos"].fillna(0).astype(int)
        # Label the view
        agg["hazard_type"] = label
        # Return the table for this view
        return agg

    # One table per hazard type
    partes = [_por_vista(df[df.hazard_type == h], h) for h in sorted(df.hazard_type.unique())]
    # Add the combined view of all hazards
    partes.append(_por_vista(df, "TODOS"))
    # Stack all views
    result = pd.concat(partes, ignore_index=True)
    # Check the maximum magnitude is between 0 and 1
    assert result["enso_magnitude_max"].between(0, 1).all(), "enso_magnitude_max fuera de [0,1]"
    # Check the mean magnitude is between 0 and 1
    assert result["enso_magnitude_mean"].between(0, 1).all(), "enso_magnitude_mean fuera de [0,1]"
    # Check the mean never exceeds the maximum
    assert (result["enso_magnitude_mean"] <= result["enso_magnitude_max"] + 1e-9).all(), \
        "el promedio nunca puede superar el máximo — error de cálculo si ocurre"
    # Return the final table
    return result

In [ ]:
# Log the start of the canton-level calculation
logger.info("Calculando Índice de Magnitud ENSO a nivel cantón...")
# Compute the magnitude index at canton level
mag_canton = ejecutar_nivel_magnitud(he, CANTON_COLS)
# Log the start of the province-level calculation
logger.info("Calculando Índice de Magnitud ENSO a nivel provincia...")
# Compute the magnitude index at province level
mag_provincia = ejecutar_nivel_magnitud(he, PROVINCE_COLS)
# Log the number of rows produced
logger.info("Cantón: %d filas | Provincia: %d filas", len(mag_canton), len(mag_provincia))

## 4. Validación

**English:** 4. Validation

In [ ]:
# Number of hazard types in the data
n_hazard_types = he.hazard_type.nunique()
# Check every canton appears in every view
assert len(mag_canton) == 221 * (n_hazard_types + 1), "cobertura incompleta a nivel cantón"
# Check every province appears in every view
assert len(mag_provincia) == 24 * (n_hazard_types + 1), "cobertura incompleta a nivel provincia"

# Con el criterio ampliado (union ICEN+ONI) si se alcanza 1.0 -- ver nota de seccion 1 (episodio 2015-2016, ONI "Muy Fuerte").
# Check the canton maximum never exceeds 1
assert mag_canton["enso_magnitude_max"].max() <= 1.0 + 1e-9, "la magnitud combinada no puede superar el techo de la escala ordinal (1.0)"
# Check the canton mean never exceeds 1
assert mag_canton["enso_magnitude_mean"].max() <= 1.0 + 1e-9, "idem para el promedio"

# Combined hazard view at canton level
vista_todos = mag_canton[mag_canton.hazard_type == "TODOS"]

# Log the coverage check
logger.info(
    "Distintos valores en vista TODOS -- máximo: %d/221 | promedio: %d/221",
    vista_todos["enso_magnitude_max"].nunique(), vista_todos["enso_magnitude_mean"].nunique(),
)
# Print a header for the maximum distribution
print("Distribución enso_magnitude_max (vista TODOS):")
# Print how many cantons have each maximum value
print(vista_todos["enso_magnitude_max"].value_counts().sort_index())
# Print an empty line
print()
# Print a header for the mean distribution
print("Distribución enso_magnitude_mean (vista TODOS) -- resumen estadístico:")
# Print summary statistics of the mean
print(vista_todos["enso_magnitude_mean"].describe())
# Print an empty line
print()

# Ten cantons with the highest mean magnitude
top10 = vista_todos.sort_values("enso_magnitude_mean", ascending=False).head(10)
# Print a header for the ranking
print("Top 10 cantones por enso_magnitude_mean:")
# Print the ranking
print(top10[["province_name_official", "canton_name_official", "enso_magnitude_max",
             "enso_magnitude_mean", "n_eventos"]].to_string(index=False))

## 5. Exportación

**English:** 5. Export

In [ ]:
# Tables to export
tablas = {
    "enso_magnitude_index_canton": mag_canton,
    "enso_magnitude_index_provincia": mag_provincia,
}

# Loop over the tables
for nombre, df in tablas.items():
    # Save as Parquet
    df.to_parquet(processed_dir / f"{nombre}.parquet", index=False)
    # Save as CSV
    df.to_csv(processed_dir / f"{nombre}.csv", index=False)
    # Log each exported table
    logger.info("Exportado: %s (%d filas)", nombre, len(df))

# Print the export folder
print(f"Tablas exportadas a {processed_dir} :")
# Loop over the exported tables
for nombre in tablas:
    # Print each table name
    print(f" - {nombre}")

## 6. Próximos pasos

- **Fase 8 — HCI compuesto**: ya están listos los cuatro insumos —
  Severidad (Fase 4), Recurrencia (Fase 5), Persistencia (Fase 6) y
  Magnitud ENSO (esta fase, con `enso_magnitude_max` y
  `enso_magnitude_mean` ambas disponibles). Combinar con
  `HCI = 0.35·Recurrencia + 0.30·Severidad + 0.20·Persistencia +
  0.15·Magnitud ENSO` (Fase 0 §9), sujeto al análisis de sensibilidad ya
  pedido explícitamente en el diseño original para estos pesos, más los
  parámetros internos ya señalados en cada fase: Fase 4 §6 (outliers de
  min-max), Fase 5 §10 (definición de "evento ENSO" por episodios), Fase 6
  §4.2 (pesos iguales de persistencia), y **la elección entre
  `enso_magnitude_max` y `enso_magnitude_mean` de esta fase, todavía
  pendiente**.
- Clasificación final (Muy Baja a Muy Alta) sobre el HCI compuesto,
  usando cuantiles, quiebres naturales o percentiles (Fase 0 §7) — pendiente
  de decidir el método definitivo.